# FloodShift 04H-CAL — Final Validation-Only Hierarchical Type Calibration

This notebook performs **no training**.

It implements the single final calibration experiment motivated by the completed 04H run:

- 04H epoch 4 validation:
  - mIoU = **0.7932**
  - non-flood IoU = **0.9765**
  - permanent-water IoU = **0.8191**
  - temporary-flood IoU = **0.5839**
- Scientific gate:
  - mIoU >= **0.790**
  - permanent-water IoU >= **0.820**
  - temporary-flood IoU >= **0.570**

04H missed the permanent-water gate by only **0.0009**, while temporary flood retained substantial headroom.

The 04H hierarchy defines

\[
L_T = z_{\mathrm{temporary}} - z_{\mathrm{permanent}}
\]

for water-type discrimination. This notebook tests one fixed scalar calibration

\[
L_T' = L_T + b
\]

using only the predeclared validation candidates:

```text
b ∈ {0.000, -0.025, -0.050, -0.075, -0.100, -0.150}
```

A negative bias slightly favors **permanent water** over **temporary flood**.

## Scientific constraints

- **No training**
- **No backbone/head updates**
- **No test Dataset or DataLoader**
- **No Pakistan/Spain test evaluation**
- Same frozen preprocessing and validation split
- Same exact 4-way TTA protocol as 04H
- Candidate set is fixed before execution
- Bias `0.0` must first reproduce the archived 04H validation result
- Selection is validation-only
- If no candidate satisfies the fixed gates, calibration is rejected and segmentation tuning stops

## Required Kaggle inputs

1. The same permanently frozen preprocessing output used by 04H, containing:
   - `floodshift_preprocessing_manifest_v1.csv`
   - `floodshift_preprocessing_GLOBAL_LOCK_v1.json`
   - `optical_10band_train_normalization_exact.csv`
   - `train_raw_sar_scene_statistics.csv`
   - all raster/reliability files referenced by the manifest

2. The **full selected 04H epoch-4 checkpoint**:
   - `FloodShift_04H_best_mIoU.pt`

The notebook checks that the checkpoint:
- is based on the exact archived 04E SHA256;
- corresponds to epoch 4 / the selected best-mIoU model;
- carries validation metrics consistent with the completed 04H run.

You do **not** need 04F or 04G checkpoints for this calibration notebook.


In [ ]:
# ============================================================
# CELL: P11-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P10-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E/04F-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P11-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P10-C01.")

In [ ]:
# ============================================================
# CELL: P11-C01 — ENVIRONMENT + REPRODUCIBILITY
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, time, math, random, hashlib, zipfile
from pathlib import Path

import numpy as np
import scipy
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_04H_CAL")
BOOTSTRAP_ROOT = OUTPUT_ROOT / "bootstrap"
METRIC_ROOT = OUTPUT_ROOT / "metrics"

for p in [OUTPUT_ROOT, BOOTSTRAP_ROOT, METRIC_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P11-C01 — 04H-CAL ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("TerraTorch  :", getattr(terratorch, "__version__", "unknown"))
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL: P11-C02 — LOCATE FROZEN PREPROCESSING + EXACT 04H CHECKPOINT
# ============================================================
_file_cache = {}

def locate_by_saved_path(saved_path_or_name, required=True):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    key = str(saved_path_or_name)

    if key in _file_cache:
        return _file_cache[key]

    matches = []
    for root in [INPUT_ROOT, BOOTSTRAP_ROOT]:
        if root.exists():
            matches.extend(root.rglob(filename))

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not locate '{filename}' under /kaggle/input."
            )
        return None

    selected = sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]
    _file_cache[key] = selected
    return selected

# Optional extraction if the user attached a full 04H archive manually.
# The ordinary C13 compact archive does NOT contain the full best checkpoint,
# so the full FloodShift_04H_best_mIoU.pt remains the preferred input.
for archive_name in [
    "FloodShift_04H_FULL_ARCHIVE.zip",
]:
    matches = list(INPUT_ROOT.rglob(archive_name))
    if matches:
        archive = sorted(matches, key=lambda p: len(str(p)))[0]
        print("Extracting:", archive)
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(BOOTSTRAP_ROOT)

FINAL_MANIFEST_PATH = locate_by_saved_path("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate_by_saved_path("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate_by_saved_path("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate_by_saved_path("train_raw_sar_scene_statistics.csv")

CHECKPOINT_04H = locate_by_saved_path("FloodShift_04H_best_mIoU.pt")

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

SHA_04H = sha256_file(CHECKPOINT_04H)

print("=" * 112)
print("P11-C02 — INPUTS FOUND")
print("=" * 112)
print("Manifest       :", FINAL_MANIFEST_PATH)
print("Global lock    :", GLOBAL_LOCK_PATH)
print("Optical stats  :", OPTICAL_NORM_PATH)
print("SAR stats      :", SAR_SCENE_STATS_PATH)
print("04H checkpoint :", CHECKPOINT_04H)
print("04H SHA256     :", SHA_04H)
print("Lock           :", preprocessing_lock["status"])
print("P11-C02 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P11-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P11-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

In [ ]:
# ============================================================
# CELL: P11-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [ ]:
# ============================================================
# CELL: P11-C05 — VALIDATION LOADER ONLY
#
# Frozen test split is asserted but deliberately never wrapped in a Dataset.
# ============================================================
VAL_BATCH_SIZE = 2
NUM_WORKERS = 2

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

val_dataset = FixedEvalDataset(val_df, train_mode=False)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    persistent_workers=(NUM_WORKERS > 0),
)

batch = next(iter(val_loader))

print("=" * 112)
print("P11-C05 — VALIDATION-ONLY DATA CHECK")
print("=" * 112)
print("Validation scenes / tiles:", len(val_df), "/", len(val_dataset))
print("Frozen test scenes        :", len(test_df), "(LOCKED; no Dataset/DataLoader constructed)")
print("SAR                       :", tuple(batch["sar"].shape))
print("Optical                   :", tuple(batch["optical"].shape))
print("Reliability               :", tuple(batch["reliability"].shape))
print("Target                    :", tuple(batch["target"].shape))

assert batch["sar"].shape[1:] == (2, 256, 256)
assert batch["optical"].shape[1:] == (10, 256, 256)
assert len(test_df) == 58
print("P11-C05 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P11-C06 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits

In [ ]:
# ============================================================
# CELL: P11-C07 — REBUILD 04H + LOAD EXACT EPOCH-4 CHECKPOINT
# ============================================================
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"

class HierarchicalResidualHead(nn.Module):
    def __init__(self, in_channels=16, hidden=48, max_delta=1.50):
        super().__init__()
        self.max_delta = float(max_delta)
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, base_prob, sar, optical, reliability):
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        raw = self.net(x).float()
        return self.max_delta * torch.tanh(raw / self.max_delta)

class FloodShift04H(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = HierarchicalResidualHead()
        for p in self.base.parameters():
            p.requires_grad = False
        self.base.eval()

    @property
    def backbone(self):
        return self.base.backbone

    def train(self, mode=True):
        super().train(mode)
        self.base.eval()
        self.head.train(mode)
        return self

    def forward(self, sar, optical, reliability, return_aux=False):
        with torch.no_grad():
            base_out = self.base(sar, optical, reliability, return_aux=True)
            base_logits = base_out["logits"].float()
            base_prob = torch.softmax(base_logits, dim=1)

        correction = self.head(
            base_prob.to(sar.dtype),
            sar,
            optical,
            reliability,
        )
        delta_water = correction[:, 0]
        delta_type = correction[:, 1]

        base_water_logit = torch.logsumexp(base_logits[:, 1:3], dim=1) - base_logits[:, 0]
        base_type_logit = base_logits[:, 2] - base_logits[:, 1]

        water_logit = base_water_logit + delta_water
        type_logit = base_type_logit + delta_type

        p_water = torch.sigmoid(water_logit)
        p_temp_given_water = torch.sigmoid(type_logit)

        p0 = 1.0 - p_water
        p1 = p_water * (1.0 - p_temp_given_water)
        p2 = p_water * p_temp_given_water

        probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
        log_probs = probs.log()

        if return_aux:
            return {
                "logits": log_probs,
                "probs": probs,
                "base_logits": base_logits,
                "base_prob": base_prob,
                "base_water_logit": base_water_logit,
                "base_type_logit": base_type_logit,
                "water_logit": water_logit,
                "type_logit": type_logit,
                "delta_water": delta_water,
                "delta_type": delta_type,
            }
        return log_probs

payload = torch.load(CHECKPOINT_04H, map_location="cpu", weights_only=False)

assert isinstance(payload, dict), "Unexpected 04H checkpoint format."
assert payload.get("base_sha256") == EXPECTED_04E_SHA256, \
    "04H does not descend from the exact archived 04E checkpoint."

selected_epoch = int(payload.get("epoch", -1))
selected_tag = payload.get("tag", "")
stored_val = payload.get("validation_metrics", {})

print("Stored epoch :", selected_epoch)
print("Stored tag   :", selected_tag)
if stored_val:
    print("Stored mIoU  :", stored_val.get("mIoU"))
    print("Stored perm  :", stored_val.get("permanent_water_IoU"))
    print("Stored temp  :", stored_val.get("temporary_flood_IoU"))

assert selected_epoch == 4, f"Expected the selected epoch-4 04H checkpoint, got epoch {selected_epoch}."
assert selected_tag == "best_mIoU", f"Expected best_mIoU checkpoint, got tag={selected_tag!r}."

if stored_val:
    assert abs(float(stored_val["mIoU"]) - 0.7932) < 0.002
    assert abs(float(stored_val["permanent_water_IoU"]) - 0.8191) < 0.002
    assert abs(float(stored_val["temporary_flood_IoU"]) - 0.5839) < 0.002

base = FloodShiftTerraMindRIFT(num_classes=NUM_CLASSES, pretrained_backbone=False)
model = FloodShift04H(base)

state = payload["state_dict"]
missing, unexpected = model.load_state_dict(state, strict=False)
assert not missing, f"04H missing keys: {missing[:10]}"
assert not unexpected, f"04H unexpected keys: {unexpected[:10]}"

bad = [
    n for n, t in model.state_dict().items()
    if torch.is_floating_point(t) and not torch.isfinite(t).all()
]
assert not bad, f"Non-finite tensors in 04H checkpoint: {bad[:10]}"

model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False

print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("P11-C07 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P11-C08 — METRICS + CALIBRATED 4-WAY TTA
# ============================================================
def confusion_from_predictions(pred, target):
    valid = target != IGNORE_INDEX
    target_v = target[valid].long()
    pred_v = pred[valid].long()
    indices = NUM_CLASSES * target_v + pred_v
    return torch.bincount(
        indices,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics_from_confusion(cm):
    if torch.is_tensor(cm):
        cm = cm.detach().cpu().numpy().astype(np.float64)
    else:
        cm = np.asarray(cm, dtype=np.float64)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    den_iou = tp + fp + fn
    iou = np.divide(tp, den_iou, out=np.full_like(tp, np.nan), where=den_iou > 0)

    den_dice = 2 * tp + fp + fn
    dice = np.divide(2 * tp, den_dice, out=np.full_like(tp, np.nan), where=den_dice > 0)

    out = {
        "mIoU": float(np.nanmean(iou)),
        "macro_dice": float(np.nanmean(dice)),
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(CLASS_NAMES):
        out[f"{name}_IoU"] = float(iou[i])
        out[f"{name}_Dice"] = float(dice[i])

    return out

def compact_metrics(m):
    return {
        "mIoU": round(m["mIoU"], 4),
        "non_flood": round(m["non_flood_IoU"], 4),
        "permanent": round(m["permanent_water_IoU"], 4),
        "temporary": round(m["temporary_flood_IoU"], 4),
    }

def _tta_transform(tensor, mode):
    if mode == "identity":
        return tensor
    if mode == "hflip":
        return torch.flip(tensor, dims=[-1])
    if mode == "vflip":
        return torch.flip(tensor, dims=[-2])
    if mode == "rot180":
        return torch.rot90(tensor, k=2, dims=(-2, -1))
    raise ValueError(mode)

def _tta_inverse(tensor, mode):
    return _tta_transform(tensor, mode)

def calibrated_log_probs_from_aux(aux, type_bias):
    """
    Apply ONLY the scalar water-type calibration:
        L_type' = L_type + type_bias

    Water-vs-background logit is unchanged.
    """
    water_logit = aux["water_logit"].float()
    type_logit = aux["type_logit"].float() + float(type_bias)

    p_water = torch.sigmoid(water_logit)
    p_temp_given_water = torch.sigmoid(type_logit)

    p0 = 1.0 - p_water
    p1 = p_water * (1.0 - p_temp_given_water)
    p2 = p_water * p_temp_given_water

    probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
    return probs.log()

@torch.inference_mode()
def evaluate_fixed_biases(model_to_eval, loader, candidate_biases):
    """
    One set of expensive model forwards per TTA branch.
    All fixed scalar candidates are evaluated from the same outputs.
    """
    cms = {
        float(b): torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
        for b in candidate_biases
    }

    modes = ["identity", "hflip", "vflip", "rot180"]
    model_to_eval.eval()

    for bi, batch in enumerate(loader, start=1):
        sar = batch["sar"].to(DEVICE, non_blocking=True)
        opt = batch["optical"].to(DEVICE, non_blocking=True)
        rel = batch["reliability"].to(DEVICE, non_blocking=True)
        y = batch["target"].to(DEVICE, non_blocking=True)

        totals = {float(b): None for b in candidate_biases}

        for mode in modes:
            sar_t = _tta_transform(sar, mode)
            opt_t = _tta_transform(opt, mode)
            rel_t = _tta_transform(rel, mode)

            with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                aux = model_to_eval(
                    sar_t,
                    opt_t,
                    rel_t,
                    return_aux=True,
                )

            for b in candidate_biases:
                b = float(b)
                log_probs = calibrated_log_probs_from_aux(aux, b)
                log_probs = _tta_inverse(log_probs, mode).float()
                totals[b] = log_probs if totals[b] is None else totals[b] + log_probs

        for b in candidate_biases:
            b = float(b)
            avg_log_probs = totals[b] / len(modes)
            pred = avg_log_probs.argmax(1)
            cms[b] += confusion_from_predictions(pred, y).cpu()

        if bi == 1 or bi % 30 == 0 or bi == len(loader):
            print(f"[{bi:03d}/{len(loader)}] validation")

    return {
        b: metrics_from_confusion(cm)
        for b, cm in cms.items()
    }

print("P11-C08 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P11-C09 — FIXED VALIDATION-ONLY CALIBRATION SWEEP
# ============================================================
# PREDECLARED. Do not edit after seeing results.
TYPE_BIAS_CANDIDATES = [
     0.000,
    -0.025,
    -0.050,
    -0.075,
    -0.100,
    -0.150,
]

PROMOTION_MIN_MIOU = 0.790
PROMOTION_MIN_PERM_IOU = 0.820
PROMOTION_MIN_TEMP_IOU = 0.570

REF_04H = {
    "mIoU": 0.7932,
    "non_flood": 0.9765,
    "permanent": 0.8191,
    "temporary": 0.5839,
}

REF_04G = {
    "mIoU": 0.7953,
    "non_flood": 0.9749,
    "permanent": 0.8339,
    "temporary": 0.5772,
}

# Single-model preference rule retained from 04H, plus no regression below
# the 04G temporary-flood IoU if we are going to replace the ensemble.
SINGLE_MODEL_MIN_MIOU = REF_04G["mIoU"] - 0.003  # 0.7923
SINGLE_MODEL_MIN_TEMP = REF_04G["temporary"]      # 0.5772

print("=" * 112)
print("P11-C09 — PREDECLARED CALIBRATION")
print("=" * 112)
print("Type biases:", TYPE_BIAS_CANDIDATES)
print("Gate       :",
      f"mIoU>={PROMOTION_MIN_MIOU:.3f}, "
      f"perm>={PROMOTION_MIN_PERM_IOU:.3f}, "
      f"temp>={PROMOTION_MIN_TEMP_IOU:.3f}")
print("Preferred single-model rule:",
      f"mIoU>={SINGLE_MODEL_MIN_MIOU:.4f}, "
      f"temp>={SINGLE_MODEL_MIN_TEMP:.4f}, "
      f"perm>={PROMOTION_MIN_PERM_IOU:.3f}")

start = time.time()
all_results = evaluate_fixed_biases(
    model,
    val_loader,
    TYPE_BIAS_CANDIDATES,
)
print("Calibration validation minutes:", round((time.time() - start) / 60, 2))

# Bias 0 must reproduce the exact selected 04H checkpoint.
zero = all_results[0.0]
print("\nBias 0 reproduction:", compact_metrics(zero))

assert abs(zero["mIoU"] - REF_04H["mIoU"]) < 0.0010, \
    "Bias 0 failed to reproduce 04H mIoU."
assert abs(zero["permanent_water_IoU"] - REF_04H["permanent"]) < 0.0010, \
    "Bias 0 failed to reproduce 04H permanent-water IoU."
assert abs(zero["temporary_flood_IoU"] - REF_04H["temporary"]) < 0.0010, \
    "Bias 0 failed to reproduce 04H temporary-flood IoU."

rows = []
for b in TYPE_BIAS_CANDIDATES:
    m = all_results[float(b)]
    gate_pass = (
        m["mIoU"] >= PROMOTION_MIN_MIOU
        and m["permanent_water_IoU"] >= PROMOTION_MIN_PERM_IOU
        and m["temporary_flood_IoU"] >= PROMOTION_MIN_TEMP_IOU
    )
    preferred = (
        gate_pass
        and m["mIoU"] >= SINGLE_MODEL_MIN_MIOU
        and m["temporary_flood_IoU"] >= SINGLE_MODEL_MIN_TEMP
    )

    rows.append({
        "type_bias": float(b),
        "mIoU": m["mIoU"],
        "non_flood_IoU": m["non_flood_IoU"],
        "permanent_water_IoU": m["permanent_water_IoU"],
        "temporary_flood_IoU": m["temporary_flood_IoU"],
        "gate_pass": gate_pass,
        "single_model_preferred": preferred,
    })

results_df = pd.DataFrame(rows)
print("\nFULL FIXED CANDIDATE RESULTS")
display(
    results_df.style.format({
        "type_bias": "{:.3f}",
        "mIoU": "{:.6f}",
        "non_flood_IoU": "{:.6f}",
        "permanent_water_IoU": "{:.6f}",
        "temporary_flood_IoU": "{:.6f}",
    })
)

results_csv = METRIC_ROOT / "FloodShift_04H_CAL_candidates.csv"
results_df.to_csv(results_csv, index=False)
print("Saved:", results_csv)
print("P11-C09 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P11-C10 — FINAL CALIBRATION SELECTION
# ============================================================
gate_rows = results_df[results_df["gate_pass"]].copy()

if len(gate_rows) == 0:
    selected_bias = 0.0
    selected_metrics = all_results[0.0]
    decision = "CALIBRATION_REJECTED"
    single_model_preferred = False

    print("=" * 112)
    print("P11-C10 — DECISION")
    print("=" * 112)
    print("No predeclared bias satisfied all scientific gates.")
    print("DECISION: CALIBRATION REJECTED ❌")
    print("Keep original 04H checkpoint unchanged.")
    print("Do not launch another segmentation experiment automatically.")

else:
    # Validation selection:
    # 1. satisfy ALL fixed scientific gates;
    # 2. highest mIoU;
    # 3. if numerically tied, prefer the smallest |bias| (most conservative).
    gate_rows["abs_bias"] = gate_rows["type_bias"].abs()
    gate_rows = gate_rows.sort_values(
        by=["mIoU", "abs_bias"],
        ascending=[False, True],
    )

    winner = gate_rows.iloc[0]
    selected_bias = float(winner["type_bias"])
    selected_metrics = all_results[selected_bias]

    single_model_preferred = bool(
        selected_metrics["mIoU"] >= SINGLE_MODEL_MIN_MIOU
        and selected_metrics["temporary_flood_IoU"] >= SINGLE_MODEL_MIN_TEMP
        and selected_metrics["permanent_water_IoU"] >= PROMOTION_MIN_PERM_IOU
    )

    decision = (
        "04H_CAL_SINGLE_MODEL_PREFERRED"
        if single_model_preferred
        else "04H_CAL_GATE_PASS"
    )

    print("=" * 112)
    print("P11-C10 — DECISION")
    print("=" * 112)
    print("Selected type bias :", selected_bias)
    print("04H uncalibrated   :", REF_04H)
    print("04G reference      :", REF_04G)
    print("04H calibrated     :", compact_metrics(selected_metrics))
    print()
    print("Δ vs 04H:")
    print("  mIoU      :", selected_metrics["mIoU"] - REF_04H["mIoU"])
    print("  permanent :", selected_metrics["permanent_water_IoU"] - REF_04H["permanent"])
    print("  temporary :", selected_metrics["temporary_flood_IoU"] - REF_04H["temporary"])
    print()

    if single_model_preferred:
        print("DECISION: 04H-CAL SINGLE MODEL PREFERRED ✅")
        print("The calibrated single model clears the fixed class gates,")
        print("stays within 0.003 mIoU of 04G, and retains at least 04G temporary-flood IoU.")
    else:
        print("DECISION: 04H-CAL PASSES THE ORIGINAL SCIENTIFIC GATE ✅")
        print("but it does not satisfy the stronger single-model preference rule.")

print()
print("🔒 FROZEN GEOGRAPHIC TEST REMAINS LOCKED.")
print("P11-C10 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P11-C11 — EXPORT CALIBRATION RECORD + OPTIONAL STANDALONE CHECKPOINT
# ============================================================
record = {
    "experiment": "FloodShift-04H-CAL",
    "operation": "validation-only scalar calibration of temporary-vs-permanent type logit",
    "training_performed": False,
    "source_04h_checkpoint": str(CHECKPOINT_04H),
    "source_04h_sha256": SHA_04H,
    "source_epoch": selected_epoch,
    "source_tag": selected_tag,
    "source_04e_sha256": EXPECTED_04E_SHA256,
    "candidate_biases": TYPE_BIAS_CANDIDATES,
    "selection_rule": (
        "among fixed candidates satisfying all scientific gates, "
        "select highest validation mIoU; numerical ties prefer smallest |bias|"
    ),
    "reference_04h": REF_04H,
    "reference_04g": REF_04G,
    "gates": {
        "mIoU": PROMOTION_MIN_MIOU,
        "permanent_water_IoU": PROMOTION_MIN_PERM_IOU,
        "temporary_flood_IoU": PROMOTION_MIN_TEMP_IOU,
    },
    "single_model_preference": {
        "min_mIoU": SINGLE_MODEL_MIN_MIOU,
        "min_temporary_flood_IoU": SINGLE_MODEL_MIN_TEMP,
        "min_permanent_water_IoU": PROMOTION_MIN_PERM_IOU,
    },
    "candidate_results": rows,
    "selected_type_bias": selected_bias,
    "selected_metrics": selected_metrics,
    "decision": decision,
    "test_status": "LOCKED_NOT_EVALUATED",
}

record_path = OUTPUT_ROOT / "FloodShift_04H_CAL_decision_record.json"
with open(record_path, "w") as f:
    json.dump(record, f, indent=2)

# Save a tiny inference calibration artifact in every case.
config_pt = OUTPUT_ROOT / "FloodShift_04H_CAL_config.pt"
torch.save(
    {
        "experiment": "FloodShift-04H-CAL",
        "source_04h_checkpoint_sha256": SHA_04H,
        "source_04e_sha256": EXPECTED_04E_SHA256,
        "type_bias": float(selected_bias),
        "validation_metrics": selected_metrics,
        "decision": decision,
    },
    config_pt,
)

# If the stronger single-model preference rule is satisfied, also save
# one standalone checkpoint containing the original 04H weights plus
# the fixed inference calibration scalar.
standalone_path = None
if single_model_preferred:
    standalone_path = OUTPUT_ROOT / "FloodShift_04H_CALIBRATED_FULL.pt"
    torch.save(
        {
            "experiment": "FloodShift-04H-CAL",
            "state_dict": model.state_dict(),
            "type_bias": float(selected_bias),
            "validation_metrics": selected_metrics,
            "source_04h_sha256": SHA_04H,
            "source_04e_sha256": EXPECTED_04E_SHA256,
            "decision": decision,
            "test_status": "LOCKED_NOT_EVALUATED",
        },
        standalone_path,
    )

archive_path = OUTPUT_ROOT / "FloodShift_04H_CAL_ARCHIVE.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [
        record_path,
        config_pt,
        METRIC_ROOT / "FloodShift_04H_CAL_candidates.csv",
    ]:
        if Path(p).exists():
            zf.write(p, arcname=Path(p).name)

print("=" * 112)
print("P11-C11 — EXPORT")
print("=" * 112)
print("Decision record :", record_path)
print("Calibration cfg :", config_pt)
print("Candidate CSV   :", METRIC_ROOT / "FloodShift_04H_CAL_candidates.csv")
print("Compact archive :", archive_path)
print("Standalone ckpt :", standalone_path if standalone_path else "not created")
print("Selected bias   :", selected_bias)
print("Decision        :", decision)
print("Test            : LOCKED_NOT_EVALUATED")
print("P11-C11 COMPLETE ✅")


## What to send back

After the notebook finishes, send the complete outputs of:

- **P11-C09** — all six calibration candidates
- **P11-C10** — selected bias and decision
- **P11-C11** — exported artifacts

Do **not** open the frozen geographic test after this run. Once this final calibration decision is known, the next step is to freeze the segmentation inference rule and move to the robustness / uncertainty protocol.
